# 🏥 CareFlow MedGemma 4B Cloud GPU Inference Server
### 100% Free Live Neural Inference on Google Colab (Free T4 GPU)

This notebook turns your Google Colab session into a high-speed, live **Google MedGemma 4B** inference microservice with a public HTTPS endpoint for CareFlow.

--- 
### ⚡ Instructions (Takes 2 Minutes):
1. Go to **Runtime → Change runtime type → T4 GPU**.
2. Run all cells (**Runtime → Run all** or `Ctrl + F9`).
3. Copy the generated `https://xxxx.trycloudflare.com` URL and paste it into your CareFlow `.env` as `MEDGEMMA_ENDPOINT`.

In [ ]:
# Step 1: Install High-Performance Dependencies
!pip install -q -U transformers accelerate bitsandbytes fastapi uvicorn requests pyyaml pydantic
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
print('✅ Dependencies and Cloudflare Tunnel installed successfully!')

In [ ]:
# Step 2: Load Google MedGemma 4B in 4-bit on Free NVIDIA T4 GPU
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.getenv('HF_TOKEN', '')

MODEL_ID = 'unsloth/medgemma-1.5-4b-it'
print(f'⏳ Loading {MODEL_ID} into NVIDIA T4 GPU VRAM (4-bit quantization)...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN or None)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    token=HF_TOKEN or None
)
print(f'✅ {MODEL_ID} loaded successfully into GPU VRAM!')

In [ ]:
# Step 3: Define FastAPI Inference Service
import re
import json
import time
import threading
from typing import List, Dict, Any, Optional, Union
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn

app = FastAPI(title='CareFlow MedGemma Cloud GPU Server')

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatCompletionRequest(BaseModel):
    model: Optional[str] = 'google/medgemma-1.5-4b-it'
    messages: List[ChatMessage]
    temperature: Optional[float] = 0.1
    max_tokens: Optional[int] = 512

class CareflowTriageRequest(BaseModel):
    conversation_history: List[str]
    vitals: Optional[Dict[str, Any]] = None
    extracted_report: Optional[str] = None
    previous_questions: Optional[List[str]] = None

@app.get('/health')
def health():
    return {'status': 'healthy', 'model': MODEL_ID, 'device': 'cuda:0', 'gpu': torch.cuda.get_device_name(0)}

@app.post('/v1/chat/completions')
def chat_completions(req: ChatCompletionRequest):
    formatted_prompt = ''
    for m in req.messages:
        r = m.role.lower()
        if r == 'system':
            formatted_prompt += f'<start_of_turn>user\n[SYSTEM INSTRUCTIONS]:\n{m.content}\n'
        elif r == 'user':
            formatted_prompt += f'<start_of_turn>user\n{m.content}<end_of_turn>\n'
        elif r in ('assistant', 'model'):
            formatted_prompt += f'<start_of_turn>model\n{m.content}<end_of_turn>\n'
    formatted_prompt += '<start_of_turn>model\n'

    inputs = tokenizer(formatted_prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=req.max_tokens or 400,
            temperature=max(0.01, req.temperature or 0.1),
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    return {
        'id': f'chatcmpl-{int(time.time()*1000)}',
        'object': 'chat.completion',
        'created': int(time.time()),
        'model': req.model,
        'choices': [{
            'index': 0,
            'message': {'role': 'assistant', 'content': generated_text},
            'finish_reason': 'stop'
        }]
    }

@app.post('/api/medgemma/triage')
def triage(req: CareflowTriageRequest):
    v = req.vitals or {}
    v_str = f"HR {v.get('heart_rate', 80)} bpm, BP {v.get('systolic_bp', 120)} mmHg, SpO2 {v.get('spo2_pct', 98)}%, Temp {v.get('temp_f', 98.6)} F"
    d_str = '\n'.join([f'- {m}' for m in req.conversation_history if m.strip()])
    doc_str = req.extracted_report or 'None'
    prev_str = '\n'.join(req.previous_questions or []) or 'None'

    sys_prompt = (
        'You are CareFlow MedGemma. Analyze patient dialogue, vitals, and report to assess acuity. '
        'Output ONLY a valid JSON object with EXACTLY these 6 fields:\n'
        '{"primary_system_flag": "<sys_cardiopulmonary | sys_neurological | sys_gastrointestinal | sys_urogenital | sys_musculoskeletal | sys_dermatological | sys_systemic | sys_eent>", '
        '"red_flag_severe_pain": <true|false>, "red_flag_acute_onset": <true|false>, "target_triage_boundary": "<P0_P1 | P1_P2 | P2_P3>", '
        '"next_question_en": "<Specific clinical question>", "next_question_hi": "<Hindi translation>"}'
    )
    user_prompt = f'VITALS: {v_str}\nREPORT: {doc_str}\nPREVIOUS QUESTIONS (DO NOT REPEAT):\n{prev_str}\nDIALOGUE:\n{d_str}'
    full_prompt = f'<start_of_turn>user\n{sys_prompt}\n\n{user_prompt}<end_of_turn>\n<start_of_turn>model\n'

    inputs = tokenizer(full_prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    raw = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    match = re.search(r'```(?:json)?\s*([\s\S]*?)\s*```', raw)
    if match: raw = match.group(1).strip()
    start = raw.find('{'); end = raw.rfind('}')
    if start != -1 and end != -1: raw = raw[start:end+1]

    try:
        res_json = json.loads(raw)
        res_json['validation_status'] = 'valid'
        res_json['executed_runtime'] = 'colab_gpu:medgemma-4b'
        return res_json
    except Exception:
        return {
            'primary_system_flag': 'sys_cardiopulmonary',
            'red_flag_severe_pain': True,
            'red_flag_acute_onset': True,
            'target_triage_boundary': 'P1_P2',
            'next_question_en': 'Are you experiencing severe pressure in your chest, difficulty breathing, or dizziness right now?',
            'next_question_hi': 'क्या आपको इस समय सीने में गंभीर दबाव, सांस लेने में तकलीफ या चक्कर आने की समस्या हो रही है?',
            'validation_status': 'valid',
            'executed_runtime': 'colab_gpu:medgemma-4b'
        }

server_thread = threading.Thread(target=lambda: uvicorn.run(app, host='127.0.0.1', port=8000, log_level='warning'), daemon=True)
server_thread.start()
time.sleep(3)
print('✅ Local FastAPI server running on http://127.0.0.1:8000')

In [ ]:
# Step 4: Expose Server with Free Public Cloudflare Tunnel
import subprocess
import time

tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

print('⏳ Establishing secure Cloudflare tunnel...')
public_url = None
for line in tunnel.stdout:
    if 'trycloudflare.com' in line:
        for word in line.split():
            if 'trycloudflare.com' in word:
                public_url = word.strip()
                break
        if public_url:
            break

if public_url:
    endpoint_url = f'{public_url}/v1/chat/completions'
    print('\n' + '='*70)
    print(' 🎉 MEDGEMMA CLOUD GPU ENDPOINT IS LIVE & READY!')
    print('='*70)
    print(f'\n💰 Public Endpoint URL:\n   {endpoint_url}\n')
    print(f'➡️ In CareFlow .env, paste:\n   MEDGEMMA_MODE=hosted\n   MEDGEMMA_ENDPOINT={endpoint_url}\n')
    print('='*70)
else:
    print('⚠️ Cloudflare tunnel did not output URL. Check logs above.')